# KHUNC

Train, compare latent storage types, and save experiments. Run the cells in order; change the settings below. Select a GPU runtime in Colab.

In [ ]:
import importlib
import sys
from pathlib import Path

IN_COLAB = Path("/content").is_dir()
if IN_COLAB:
    !test -d /content/KHUNC/.git || git clone https://github.com/KadirKess/KHUNC.git /content/KHUNC
    !git -C /content/KHUNC pull --ff-only
    %cd /content/KHUNC
    !{sys.executable} -m pip install -q uv
    !{sys.executable} -m uv pip install --python "{sys.executable}" -e ".[notebook]"

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()

In [ ]:
import json
import shutil
import tempfile
from dataclasses import replace

import torch

from khunc import (
    Codec,
    Experiment,
    ModelConfig,
    QuantizationConfig,
    TrainConfig,
    compare_quantization,
    download_imagenette,
    download_kodak,
    evaluate,
    inspect_compressed,
    load_image,
    load_model,
    save_image,
    test_model,
    train,
)
from khunc.plotting import plot_quantization, plot_reconstruction, plot_training

use_drive = True
if IN_COLAB and use_drive:
    from google.colab import drive

    drive.mount("/content/drive")

output_dir = (
    Path("/content/drive/MyDrive/KHUNC/experiments")
    if IN_COLAB and use_drive
    else REPO / "experiments"
)
data_dir = REPO / "data"
print(
    "Device:",
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
)

### Settings

`latent_dtype` controls what is saved in the compressed file. `training_precision` controls training arithmetic independently. `image_size` sets the training crop size and the validation/test resize size; Kodak keeps its original resolution.

For integer experiments, [`ste`](https://arxiv.org/abs/1308.3432) rounds forward and passes the gradient straight through; [`noise`](https://arxiv.org/abs/1611.01704) adds noise up to half a quantization step during training. Both are approximations, and validation always uses actual rounding.

Training uses random square crops at varied scales, flips, and mild color changes. The augmented image is also the reconstruction target. Validation and test images receive only deterministic resizing; set `augment=False` to use resize-only training.

In [ ]:
name = "float32-augmented"
latent_dtype = "float32"  # float32, float16, bfloat16, or uint8
quantization_method = "none"  # none, ste, or noise; noise requires uint8
quantization_bits = 8  # 1 to 8; only used for uint8
training_precision = "bf16-mixed"  # 32-true, 16-mixed, bf16-mixed; check GPU support

encoder_up_layers = 2  # 0 to 4; decoder uses 4 minus this value
latent_channels = 3
base_channels = 32
image_size = 384  # Multiple of 16; greater than 160 for MS-SSIM
augment = True  # False uses resize-only training
crop_scale = (0.5, 1.0)  # Min/max fraction of original image area before resizing
horizontal_flip = 0.5  # Probability; 0 disables it
vertical_flip = 0.5  # Probability; 0 disables it
color_jitter = 0.1  # Mild brightness, contrast, saturation changes; 0 disables them
batch_size = 8
epochs = 50
learning_rate = 1e-3  # Try 1e-4 when fine-tuning
loss = "ms_ssim"  # ms_ssim, mse, or ms_ssim_mse
mse_weight = 0.1  # Only used for ms_ssim_mse
validation_fraction = 0.1
seed = 42
num_workers = 2  # Use 0 if worker processes cause trouble
lr_patience = 10
lr_factor = 0.1
early_stopping_patience = 20  # None disables early stopping
compile_model = True

run_training = True  # False loads checkpoint for testing and evaluation
checkpoint = None  # Path to saved weights for fine-tuning or evaluation
resume_from = None  # Path to last.ckpt; epochs is the total target when resuming

In [ ]:
config = TrainConfig(
    name=name,
    data_dir=data_dir / "imagenette2",
    output_dir=output_dir,
    model=ModelConfig(
        latent_channels=latent_channels,
        base_channels=base_channels,
        encoder_up_layers=encoder_up_layers,
    ),
    quantization=QuantizationConfig(
        storage=latent_dtype,
        bits=quantization_bits,
        training=quantization_method,
    ),
    augment=augment,
    crop_scale=crop_scale,
    horizontal_flip=horizontal_flip,
    vertical_flip=vertical_flip,
    color_jitter=color_jitter,
    image_size=image_size,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=learning_rate,
    loss=loss,
    mse_weight=mse_weight,
    validation_fraction=validation_fraction,
    seed=seed,
    num_workers=num_workers,
    precision=training_precision,
    lr_scheduler_patience=lr_patience,
    lr_scheduler_factor=lr_factor,
    early_stopping_patience=early_stopping_patience,
    compile_model=compile_model,
)

### Training

Train, fine-tune, resume, or load a model using the settings above.

In [ ]:
if resume_from is not None:
    previous = Experiment(Path(resume_from).expanduser().parent.parent)
    config = replace(previous.load_config(), epochs=epochs, output_dir=output_dir)
    result = train(config, resume_from=resume_from)
    model, experiment = result.model, result.experiment
elif run_training:
    dataset = download_imagenette(root=data_dir, size="full")
    config = replace(config, data_dir=dataset)
    result = train(config, checkpoint=checkpoint)
    model, experiment = result.model, result.experiment
else:
    if checkpoint is None:
        raise ValueError("Set checkpoint to evaluate without training")
    model = load_model(checkpoint)
    config = replace(config, model=model.config)
    experiment = Experiment.create(name + "-evaluation", output_dir, config)

print("Saved to:", experiment.path)

In [ ]:
metrics_path = experiment.path / "logs/training/metrics.csv"
if metrics_path.exists():
    plot_training(metrics_path, path=experiment.path / "plots/training.png")

### Imagenette testing

In [ ]:
# Imagenette's official val directory is the test set. Training has its own validation split.
if not (Path(config.data_dir) / "val").exists():
    dataset = download_imagenette(root=data_dir, size="full")
    config = replace(config, data_dir=dataset)
test_metrics = test_model(model, config)
(experiment.path / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2) + "\n")
test_metrics

### Kodak evaluation

The next cell builds SSIMULACRA2 in a temporary folder on Colab. Locally, install it with `brew install jpeg-xl` or set `metric_binary` to your executable. Scores, compressed files, reconstructions, and timings are saved in the experiment folder.

In [ ]:
metric_binary = shutil.which("ssimulacra2")
if metric_binary is None and IN_COLAB:
    build_dir = Path(tempfile.mkdtemp(prefix="khunc-ssimulacra2-"))
    !apt-get update -qq
    !apt-get install -y -qq build-essential cmake pkg-config libbrotli-dev libjpeg-dev libpng-dev
    !git clone -q --depth 1 --branch v0.12.0 --recursive --shallow-submodules \
        https://github.com/libjxl/libjxl.git {build_dir}/libjxl
    !cmake -S {build_dir}/libjxl -B {build_dir}/build \
        -DCMAKE_BUILD_TYPE=Release -DBUILD_TESTING=OFF -DJPEGXL_ENABLE_DEVTOOLS=ON
    !cmake --build {build_dir}/build --target ssimulacra2 -j 2
    metric_binary = str(build_dir / "build/tools/ssimulacra2")

if metric_binary is None or not Path(metric_binary).is_file():
    raise FileNotFoundError("Install SSIMULACRA2 or set metric_binary to its executable path")

In [ ]:
kodak = download_kodak(data_dir / "kodak")
compare_dtypes = ["float32", "float16", "bfloat16"]
compare_integer_quantization = True
compare_bits = [8, 7, 6, 5, 4]  # Integer experiments; 1 to 8 are supported

configurations = [QuantizationConfig(storage=dtype) for dtype in compare_dtypes]
if compare_integer_quantization:
    configurations += [QuantizationConfig(storage="uint8", bits=bits) for bits in compare_bits]

comparison = compare_quantization(
    model,
    kodak,
    configurations,
    experiment=experiment,
    repeats=3,
    warmup=2,
    binary=metric_binary,
)

In [ ]:
for dtype, summary in comparison.summary.items():
    print(
        f"{dtype}: SSIMULACRA2 {summary['mean_ssimulacra2']:.2f}, "
        f"{summary['mean_compressed_bytes'] / 1024:.2f} KiB, "
        f"compress {summary['mean_encode_ms']:.2f} ms, "
        f"decompress {summary['mean_decode_ms']:.2f} ms"
    )
plot_quantization(comparison, path=experiment.path / "plots/storage-comparison.png")

### Reconstruction

In [ ]:
codec = Codec(model, config.quantization)
source = kodak / "kodim01.png"
compressed_path = experiment.path / "images/kodim01.khunc"
compressed_path.write_bytes(codec.encode(source))
reconstructed = codec.decode(compressed_path)
save_image(reconstructed, experiment.path / "images/kodim01-reconstructed.png")
plot_reconstruction(load_image(source), reconstructed, path=experiment.path / "plots/kodim01.png")
inspect_compressed(compressed_path)

### External images evaluation

Set `extra_images` to a folder or list of paths to evaluate your own images.

In [ ]:
extra_images = None  # Folder path or list of image paths
if extra_images is not None:
    extra_evaluation = evaluate(
        model,
        extra_images,
        quantization=config.quantization,
        experiment=experiment,
        binary=metric_binary,
    )
    print(extra_evaluation.summary)